In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive/Code/CloudMap3/mutants/hu80/
%ls

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive/Code/CloudMap3/mutants/hu80
'Galaxy114-[hu80HA_VariantsAtHighQualityHAPositions_MQ30_WS245_DP_0_BiallelicPositions_SNPsOnly].vcf'
'Galaxy117-[hu80HA_WS245_Homozygous_variants_SubtractedHobertHawaiianHomozygousAndHeterozygous].vcf'
 hu80_WS245_annotated-variants-mapping-region.csv
 hu80_WS245_annotation.xlsx
 llm_variant_prioritization_outputs/


In [ ]:
#@title CloudMap3 • Colab notebook: Fix mapping from hu80_mapping_output.csv + knowledge‑grounded narratives with references (single block)

# =========================
# Setup (installs + imports)
# =========================
# You may comment the next line on repeat runs.
%pip -q install --upgrade "pydantic>=2,<3" pandas python-dateutil requests tabulate > /dev/null

import os, re, json, math, time, pathlib, textwrap
from typing import List, Dict, Any, Optional, Tuple
import requests
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, conint, confloat, field_validator
from datetime import datetime, timezone
from tabulate import tabulate

# =========================
# API key (Colab secrets first, then env)
# =========================
def _ensure_openai_key():
    if os.getenv("OPENAI_API_KEY"):
        return
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            print("[Info] Loaded OPENAI_API_KEY from Colab secrets.")
            return
    except Exception:
        pass
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is not set. In Colab, add it under Runtime → Secrets with the name OPENAI_API_KEY.")
_ensure_openai_key()

# Optional: NCBI key for higher E-utilities rate limits (set in Colab secrets as NCBI_API_KEY)
try:
    from google.colab import userdata  # type: ignore
    if userdata.get("NCBI_API_KEY") and not os.getenv("NCBI_API_KEY"):
        os.environ["NCBI_API_KEY"] = userdata.get("NCBI_API_KEY")
except Exception:
    pass

# =========================
# Paths
# =========================
CSV_PATH = "/content/gdrive/MyDrive/Code/CloudMap3/mutants/hu80/hu80_WS245_annotated-variants-mapping-region.csv"  # @param {type:"string"}
MAPPING_OUTPUT_PATH = "/content/gdrive/MyDrive/Code/CloudMap3/mutants/hu80/hu80_mapping_output.csv"                # @param {type:"string"}

def _with_fallback(p: str, candidates: List[str]) -> str:
    pth = pathlib.Path(p)
    if pth.exists():
        return str(pth)
    for c in candidates:
        if pathlib.Path(c).exists():
            print(f"[Info] Using fallback file: {c}")
            return c
    return str(pth)

CSV_PATH = _with_fallback(
    CSV_PATH,
    ["/mnt/data/hu80_WS245_annotated-variants-mapping-region.csv",
     "/mnt/data/hu80_WS245_annotated-variants-mapping-region (1).csv"]
)
MAPPING_OUTPUT_PATH = _with_fallback(
    MAPPING_OUTPUT_PATH,
    ["/mnt/data/hu80_mapping_output.csv"]
)

# =========================
# Model and run parameters
# =========================
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
OPENAI_MODEL    = os.getenv("OPENAI_MODEL", "gpt-5-mini")

OPENAI_TEMPERATURE = os.getenv("OPENAI_TEMPERATURE", "")   # leave empty to auto-adapt
OPENAI_SEED        = os.getenv("OPENAI_SEED", "")
OPENAI_TIMEOUT     = int(os.getenv("OPENAI_TIMEOUT", "360"))
OPENAI_MAX_RETRIES = int(os.getenv("OPENAI_MAX_RETRIES", "5"))
OPENAI_BACKOFF_BASE= float(os.getenv("OPENAI_BACKOFF_BASE", "2.0"))

TOP_K                    = 30        # how many to keep in ranking
MAX_VARIANTS_TO_SEND     = 250
FILTER_MIN_PARENTAL_RATIO= 0.40
PREFER_EMS               = True
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

OUTPUT_DIR = pathlib.Path("llm_variant_prioritization_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Networking helpers (robust)
# =========================
def _requests_get(url: str, params: Dict[str, Any], timeout: int = 30, retries: int = 3, backoff: float = 2.0):
    for i in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(backoff ** i)

def _requests_post_with_retries(url: str, headers: Dict[str, str], payload: Dict[str, Any],
                                timeout: int, max_retries: int, backoff_base: float) -> requests.Response:
    attempt = 0
    while True:
        try:
            return requests.post(url, headers=headers, json=payload, timeout=timeout)
        except (requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectTimeout,
                requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError):
            if attempt >= max_retries - 1:
                raise
            time.sleep(backoff_base ** attempt)
            attempt += 1

def _post_chat(payload: Dict[str, Any], timeout: int = OPENAI_TIMEOUT) -> str:
    headers = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}", "Content-Type": "application/json"}
    r = _requests_post_with_retries(
        f"{OPENAI_BASE_URL}/chat/completions",
        headers=headers, payload=payload, timeout=timeout,
        max_retries=OPENAI_MAX_RETRIES, backoff_base=OPENAI_BACKOFF_BASE
    )
    if r.status_code != 200:
        raise RuntimeError(f"OpenAI error {r.status_code}: {r.text[:2000]}")
    data = r.json()
    return data["choices"][0]["message"]["content"]

def openai_chat(messages: List[Dict[str, str]],
                model: str,
                response_json: bool = True,
                temperature: Optional[float] = None,
                seed: Optional[int] = None,
                timeout: int = OPENAI_TIMEOUT) -> str:
    payload: Dict[str, Any] = {"model": model, "messages": messages}
    if response_json:
        payload["response_format"] = {"type": "json_object"}
    if temperature is not None:
        payload["temperature"] = float(temperature)
    if seed is not None:
        payload["seed"] = int(seed)
    try:
        return _post_chat(payload, timeout=timeout)
    except RuntimeError as err:
        body = str(err)
        retried = False
        if ("param" in body and "temperature" in body) or ("Unsupported value" in body and "temperature" in body):
            payload.pop("temperature", None); retried = True
        if ("param" in body and "\"seed\"" in body):
            payload.pop("seed", None); retried = True
        if ("param" in body and "response_format" in body) or ("json_object" in body):
            payload.pop("response_format", None); retried = True
        if retried:
            return _post_chat(payload, timeout=timeout)
        raise

# =========================
# CSV loading + key builders
# =========================
def exists_or_raise(p: str):
    if not pathlib.Path(p).exists():
        raise FileNotFoundError(
            f"File not found: {p}\n"
            "• In Colab, upload via the left Files pane or mount Drive.\n"
            "• Then set CSV_PATH/MAPPING_OUTPUT_PATH accordingly."
        )

def coalesce(colnames: List[str], df: pd.DataFrame) -> Optional[str]:
    for c in colnames:
        if c in df.columns: return c
    return None

def normalize_chr_label(chrom: Any) -> str:
    s = str(chrom)
    s = re.sub(r'(?i)^chromosome\s*', '', s)
    s = re.sub(r'(?i)^chr', '', s)
    s = s.strip()
    map_arabic = {"1":"I","2":"II","3":"III","4":"IV","5":"V","10":"X"}
    if s in map_arabic: return map_arabic[s]
    return s.upper()

def build_variant_id(chrom, pos, ref, alt) -> str:
    c = normalize_chr_label(chrom) if pd.notna(chrom) else "NA"
    p = str(int(pos)) if pd.notna(pos) else "NA"
    r = str(ref).upper() if pd.notna(ref) else "N"
    a = str(alt).upper() if pd.notna(alt) else "N"
    return f"{c}:{p} {r}>{a}"

def short(s: Any, maxlen: int = 60) -> str:
    t = "" if s is None else str(s)
    return t if len(t) <= maxlen else (t[:maxlen-1] + "…")

def normalize_float(s: pd.Series, clamp01: bool = False) -> pd.Series:
    out = pd.to_numeric(s, errors="coerce")
    if clamp01:
        out = out.clip(lower=0.0, upper=1.0)
    return out

# =========================
# Load primary annotated CSV
# =========================
exists_or_raise(CSV_PATH)
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} variants from: {CSV_PATH}")

# Column discovery (primary)
chrom_col = coalesce(["CHROM","Chrom","chr","chrom"], df) or "CHROM"
pos_col   = coalesce(["POS","Position","POS_GRCh","pos"], df) or "POS"
ref_col   = coalesce(["REF","Ref","reference_allele","ref"], df)
alt_col   = coalesce(["ALT","Alt","alt_allele","alt"], df)

gene_col   = coalesce(["Gene_name","Gene","GENE","SYMBOL","Symbol","GeneName","ANN_Gene_Name","HGNC"], df)
effect_col = coalesce(["Effect","Consequence","ANN_Consequence","ANN_Annotation","VEP_CONSEQUENCE"], df)
impact_col = coalesce(["IMPACT","Impact","ANN_Annotation_Impact","VEP_IMPACT"], df)
aa_col     = coalesce(["HGVSp","HGVSp_Short","AAChange","Protein_change","HGVS.p"], df)
sift_col   = coalesce(["SIFT","SIFT_pred","SIFT_PRED"], df)
pphen_col  = coalesce(["PolyPhen","PolyPhen_pred","PolyPhen_PRED"], df)

# Build standardized keys for primary
df.columns = [c.strip() for c in df.columns]
df["_CHROM_N"] = df[chrom_col].apply(normalize_chr_label)
df["_POS_I"]   = pd.to_numeric(df[pos_col], errors="coerce").astype("Int64")
if ref_col: df["_REF_U"] = df[ref_col].astype(str).str.upper()
else:       df["_REF_U"] = "N"
if alt_col: df["_ALT_U"] = df[alt_col].astype(str).str.upper()
else:       df["_ALT_U"] = "N"
df["variant_id"] = df.apply(lambda r: build_variant_id(r["_CHROM_N"], r["_POS_I"], r["_REF_U"], r["_ALT_U"]), axis=1)
df["pos_key"]    = df["_CHROM_N"].astype(str) + ":" + df["_POS_I"].astype(str)

# =========================
# Load hu80_mapping_output.csv (SOURCE OF truth for parental_ratio, hawaiian_ratio, QUAL, DP, EMS_label, Change_type)
# =========================
exists_or_raise(MAPPING_OUTPUT_PATH)
mout = pd.read_csv(MAPPING_OUTPUT_PATH)
mout.columns = [c.strip() for c in mout.columns]  # strip stray spaces
# Column discovery (mapping_output)
m_chrom = coalesce(["CHROM","Chrom","chr","chrom"], mout) or "CHROM"
m_pos   = coalesce(["POS","Position","POS_GRCh","pos"], mout) or "POS"
m_ref   = coalesce(["REF","Ref","reference_allele","ref"], mout)
m_alt   = coalesce(["ALT","Alt","alt_allele","alt"], mout)

mout["_CHROM_N"] = mout[m_chrom].apply(normalize_chr_label)
mout["_POS_I"]   = pd.to_numeric(mout[m_pos], errors="coerce").astype("Int64")
if m_ref: mout["_REF_U"] = mout[m_ref].astype(str).str.upper()
else:     mout["_REF_U"] = "N"
if m_alt: mout["_ALT_U"] = mout[m_alt].astype(str).str.upper()
else:     mout["_ALT_U"] = "N"
mout["variant_id"] = mout.apply(lambda r: build_variant_id(r["_CHROM_N"], r["_POS_I"], r["_REF_U"], r["_ALT_U"]), axis=1)
mout["pos_key"]    = mout["_CHROM_N"].astype(str) + ":" + mout["_POS_I"].astype(str)

# --- DIAGNOSTIC: Why fields looked empty earlier? ---
# The primary CSV often lacks REF/ALT (becoming 'N>N'), so variant_id does not match mapping_output's variant_id.
# We therefore rely on pos_key (CHROM:POS) as the robust join key.
inter_vid = set(df["variant_id"]).intersection(set(mout["variant_id"]))
inter_pos = set(df["pos_key"]).intersection(set(mout["pos_key"]))
print(f"[Diagnostics] variant_id intersection: {len(inter_vid)} ; pos_key intersection: {len(inter_pos)}")
if len(inter_vid) < len(inter_pos):
    print("[Diagnostics] Using CHROM:POS (pos_key) to merge mapping_output fields (REF/ALT frequently missing in the annotated CSV).")

# Build fast lookup maps by both keys (variant_id and pos_key), with pos_key preferred
def _to_map(series_key: pd.Series, series_val: pd.Series) -> Dict[str, Any]:
    out = {}
    for k, v in zip(series_key, series_val):
        if pd.isna(k): continue
        out[str(k)] = v
    return out

m_parental_by_pos = _to_map(mout["pos_key"], mout.get("Parental_ratio", pd.Series([None]*len(mout))))
m_hawaiian_by_pos = _to_map(mout["pos_key"], mout.get("Hawaiian_ratio", pd.Series([None]*len(mout))))
m_qual_by_pos     = _to_map(mout["pos_key"], mout.get("QUAL", pd.Series([None]*len(mout))))
m_dp_by_pos       = _to_map(mout["pos_key"], mout.get("DP", pd.Series([None]*len(mout))))
m_ems_by_pos      = _to_map(mout["pos_key"], mout.get("EMS_label", pd.Series([None]*len(mout))))
m_change_by_pos   = _to_map(mout["pos_key"], mout.get("Change_type", pd.Series([None]*len(mout))))

# Also, fallback maps by variant_id (used if available)
m_parental_by_vid = _to_map(mout["variant_id"], mout.get("Parental_ratio", pd.Series([None]*len(mout))))
m_hawaiian_by_vid = _to_map(mout["variant_id"], mout.get("Hawaiian_ratio", pd.Series([None]*len(mout))))
m_qual_by_vid     = _to_map(mout["variant_id"], mout.get("QUAL", pd.Series([None]*len(mout))))
m_dp_by_vid       = _to_map(mout["variant_id"], mout.get("DP", pd.Series([None]*len(mout))))
m_ems_by_vid      = _to_map(mout["variant_id"], mout.get("EMS_label", pd.Series([None]*len(mout))))
m_change_by_vid   = _to_map(mout["variant_id"], mout.get("Change_type", pd.Series([None]*len(mout))))

def _map_val(vid: str, pkey: str, by_pos: Dict[str, Any], by_vid: Dict[str, Any]):
    return by_pos.get(pkey, by_vid.get(vid, None))

# =========================
# Build records (use mapping_output values via pos_key; dedup by severity)
# =========================
def effect_severity_score(effect_text: str, impact_bucket: str) -> float:
    t = (effect_text or "").lower()
    base = {"HIGH": 3.0, "MODERATE": 2.0, "LOW": 1.0, "UNKNOWN": 0.5}.get((impact_bucket or "UNKNOWN").upper(), 0.5)
    if any(k in t for k in ["stop_gained","frameshift","splice_acceptor","splice_donor","start_lost","stop_lost","exon_loss"]):
        base = max(base, 3.2)
    elif any(k in t for k in ["missense","inframe_insertion","inframe_deletion","protein_altering"]):
        base = max(base, 2.2)
    elif any(k in t for k in ["synonymous","upstream","downstream","utr","intron"]):
        base = min(base, 1.0)
    return float(base)

def impact_bucket_from(effect_text: str, impact_bucket: Optional[str]) -> str:
    if isinstance(impact_bucket, str) and impact_bucket:
        ib = impact_bucket.strip().upper()
        if ib in {"HIGH","MODERATE","LOW"}:
            return ib
    t = (effect_text or "").lower()
    if any(k in t for k in ["stop_gained","frameshift","splice_acceptor","splice_donor","start_lost","stop_lost"]): return "HIGH"
    if any(k in t for k in ["missense","inframe_insertion","inframe_deletion","protein_altering"]): return "MODERATE"
    if any(k in t for k in ["synonymous","upstream","downstream","utr","intron"]): return "LOW"
    return "UNKNOWN"

def coerce_float(x):
    try: return float(x)
    except Exception: return None

packed_all: List[Dict[str, Any]] = []
for _, row in df.iterrows():
    vid = row["variant_id"]; pkey = row["pos_key"]
    gene   = row.get(gene_col) if gene_col else None
    effect = row.get(effect_col) if effect_col else None
    ib     = impact_bucket_from(effect, row.get(impact_col) if impact_col in df.columns else None)

    # Values from mapping_output via pos_key/variant_id (robust even when REF/ALT missing in primary)
    parental_ratio = _map_val(vid, pkey, m_parental_by_pos, m_parental_by_vid)
    hawaiian_ratio = _map_val(vid, pkey, m_hawaiian_by_pos, m_hawaiian_by_vid)
    qual           = _map_val(vid, pkey, m_qual_by_pos,     m_qual_by_vid)
    depth          = _map_val(vid, pkey, m_dp_by_pos,       m_dp_by_vid)
    ems_label_raw  = _map_val(vid, pkey, m_ems_by_pos,      m_ems_by_vid)
    change_type    = _map_val(vid, pkey, m_change_by_pos,   m_change_by_vid)

    # Normalize EMS boolean + protein_change mapping as requested
    ems_bool = False
    if isinstance(ems_label_raw, str):
        ems_bool = ems_label_raw.strip().lower().startswith("ems")
    protein_change = change_type if change_type not in [None, "nan", "NaN"] else (row.get(aa_col) if aa_col else "")

    rec = {
        "variant_id": vid,
        "pos_key": pkey,
        "chrom": row["_CHROM_N"],
        "pos": int(row["_POS_I"]) if pd.notna(row["_POS_I"]) else None,
        "gene": (gene if pd.notna(gene) else "") if gene is not None else "",
        "effect": effect if pd.notna(effect) else "",
        "impact_bucket": ib,
        "protein_change": short(protein_change, 48),
        "sift": short(row.get(sift_col), 28) if sift_col else "",
        "polyphen": short(row.get(pphen_col), 28) if pphen_col else "",
        "ems": ems_bool,
        "ems_label_raw": ems_label_raw if pd.notna(ems_label_raw) else "",
        "parental_ratio": parental_ratio,
        "hawaiian_ratio": hawaiian_ratio,
        "qual": qual,
        "depth": depth,
    }
    pr = rec["parental_ratio"] if (rec["parental_ratio"] is not None and not (isinstance(rec["parental_ratio"], float) and math.isnan(rec["parental_ratio"]))) else 0.0
    ems_bonus = 0.15 if (PREFER_EMS and rec["ems"]) else 0.0
    imp_w = {"HIGH":0.45,"MODERATE":0.25,"LOW":0.05,"UNKNOWN":0.0}.get(rec["impact_bucket"],0.0)
    q = rec["qual"]; q_w = 0.0 if (q is None or (isinstance(q,float) and math.isnan(q))) else min(float(q), 200)/200.0 * 0.05
    rec["_pre_score"] = float(pr)*0.45 + ems_bonus + imp_w + q_w
    rec["_severity"]  = effect_severity_score(rec["effect"], rec["impact_bucket"])
    packed_all.append(rec)

# Deduplicate by variant_id, keeping most severe/relevant
def tie_break_key(r):
    pr = r.get("parental_ratio") or 0.0
    ems = 1.0 if r.get("ems") else 0.0
    q = r.get("qual") or 0.0
    return (r["_severity"], pr, ems, q)

packed_dedup: List[Dict[str, Any]] = []
for vid, grp in pd.DataFrame(packed_all).groupby("variant_id"):
    best = max(grp.to_dict("records"), key=tie_break_key)
    packed_dedup.append(best)
packed_df = pd.DataFrame(packed_dedup)
print(f"Prepared base records for {len(packed_df):,} variants after dedup by most severe effect.")

# =========================
# Pre-filter for LLM
# =========================
pref = []
for rec in packed_dedup:
    pr = rec.get("parental_ratio")
    if pr is not None and not (isinstance(pr,float) and math.isnan(pr)):
        if pr < FILTER_MIN_PARENTAL_RATIO:
            continue
    pref.append(rec)
if not pref:
    pref = packed_dedup
pref = sorted(pref, key=lambda r: (r["_pre_score"], r["_severity"]), reverse=True)[:MAX_VARIANTS_TO_SEND]
print(f"Will send {len(pref):,} variants to GPT‑5‑mini "
      f"(FILTER_MIN_PARENTAL_RATIO={FILTER_MIN_PARENTAL_RATIO}, MAX_VARIANTS_TO_SEND={MAX_VARIANTS_TO_SEND}).")

# =========================
# Lightweight gene-knowledge fetch (NCBI Gene summary + PubMed refs)
# =========================
def fetch_gene_knowledge(gene: str) -> Dict[str, Any]:
    gene = (gene or "").strip()
    if not gene: return {"summary":"", "refs":[]}
    # NCBI Gene summary
    api_key = os.getenv("NCBI_API_KEY","")
    params_es = {"db":"gene","retmode":"json",
                 "term": f"{gene}[Gene Name] AND Caenorhabditis elegans[Organism]"}
    if api_key: params_es["api_key"] = api_key
    summary_text = ""
    refs = []
    try:
        es = _requests_get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", params_es, timeout=20)
        js = es.json()
        ids = js.get("esearchresult",{}).get("idlist",[])
        if ids:
            gid = ids[0]
            params_sum = {"db":"gene","retmode":"json","id":gid}
            if api_key: params_sum["api_key"] = api_key
            su = _requests_get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi", params_sum, timeout=20).json()
            doc = su.get("result",{}).get(gid,{})
            summary_text = doc.get("summary","") or doc.get("description","")
    except Exception:
        pass
    # PubMed refs (relevance to dopamine/neuron fate if available)
    try:
        q = f'({gene}) AND (dopaminergic OR dopamine OR neuron) AND elegans'
        params_pm = {"db":"pubmed","retmode":"json","retmax":"3","sort":"relevance","term": q}
        if api_key: params_pm["api_key"] = api_key
        pm = _requests_get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", params_pm, timeout=20).json()
        idlist = pm.get("esearchresult",{}).get("idlist",[])[:3]
        if idlist:
            params_sm = {"db":"pubmed","retmode":"json","id":",".join(idlist)}
            if api_key: params_sm["api_key"] = api_key
            sm = _requests_get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi", params_sm, timeout=20).json()
            for pid in idlist:
                it = sm.get("result",{}).get(pid,{})
                title = it.get("title","")
                url = f"https://pubmed.ncbi.nlm.nih.gov/{pid}/"
                if title:
                    refs.append({"pmid": pid, "title": title, "url": url})
    except Exception:
        pass
    return {"summary": summary_text.strip(), "refs": refs}

# Build knowledge map for unique genes in 'pref'
unique_genes = sorted({(r.get("gene") or "").strip() for r in pref if r.get("gene")})
gene_knowledge: Dict[str, Dict[str, Any]] = {}
for g in unique_genes:
    try:
        gene_knowledge[g] = fetch_gene_knowledge(g)
    except Exception:
        gene_knowledge[g] = {"summary":"", "refs":[]}

# =========================
# LLM schemas
# =========================
class VariantRankItem(BaseModel):
    rank: conint(ge=1)
    variant_id: str
    gene: Optional[str] = ""
    causal_probability: confloat(ge=0, le=1)
    confidence: confloat(ge=0, le=1)
    rationale: str
    key_evidence: List[str] = Field(default_factory=list)

class VariantNarrative(BaseModel):
    variant_id: str
    gene: Optional[str] = ""
    narrative: str
    narrative_confidence: confloat(ge=0, le=1)

class LLMVariantOutput(BaseModel):
    summary: str
    most_likely: VariantRankItem
    ranking: List[VariantRankItem]
    annotations: List[VariantNarrative]
    @field_validator("ranking")
    @classmethod
    def non_empty_rank(cls, v):
        if not v: raise ValueError("ranking must be non-empty")
        return v
    @field_validator("annotations")
    @classmethod
    def non_empty_ann(cls, v):
        if not v: raise ValueError("annotations must be non-empty")
        return v

class LLMRankingOnly(BaseModel):
    summary: str
    most_likely: VariantRankItem
    ranking: List[VariantRankItem]

def try_parse_json(s: str) -> Dict[str, Any]:
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{[\s\S]*\}", s)
        if m:
            return json.loads(m.group(0))
        raise

# =========================
# Build prompts (allow gene knowledge + references)
# =========================
SYSTEM_PROMPT_MAIN = """You are an expert in C. elegans forward genetics and dopaminergic neuron fate.

Context:
- Screen detects loss of dopaminergic neurons via a GFP reporter.
- Goal: within the mapping interval, identify variants most likely to cause dopaminergic fate loss.

Evidence you may use:
- Structured fields: Gene_name, Effect, impact_bucket, protein_change/Change_type, EMS_label (EMS mutagen), Parental_ratio and Hawaiian_ratio (linkage proxies), QUAL, DP, SIFT/PolyPhen.
- Gene knowledge: brief summaries and PubMed links (if provided) for the gene's known/putative function in C. elegans or related biology.

Instructions:
- Rank variants by causal probability (0–1) and provide confidence (0–1).
- For EACH variant, write a 2–5 sentence narrative connecting the mutation + gene function to dopaminergic fate loss.
- **Cite evidence in brackets**, e.g., [Effect=stop_gained; Parental_ratio=0.93; EMS; GeneRef: PMID 12345, 67890].
- Prefer HIGH-impact (nonsense/frameshift/essential splice) > damaging missense > low/unknown; higher Parental_ratio increases likelihood; EMS increases prior but is not required.
- Do not fabricate references; only cite provided PubMed links. Output JSON only; no chain-of-thought."""

def format_variant_for_llm(r: Dict[str, Any]) -> Dict[str, Any]:
    def safe_round(x, nd=3):
        try: return round(float(x), nd)
        except Exception: return None
    g = (r.get("gene") or "").strip()
    gk = gene_knowledge.get(g, {"summary":"", "refs":[]})
    return {
        "variant_id": r["variant_id"],
        "Gene_name": g,
        "Effect": short(r.get("effect"), 90),
        "impact_bucket": r.get("impact_bucket","UNKNOWN"),
        "protein_change": short(r.get("protein_change",""), 48),
        "ems_label": ("EMS" if r.get("ems") else "non-EMS"),
        "Parental_ratio": safe_round(r.get("parental_ratio"), 3),
        "Hawaiian_ratio": safe_round(r.get("hawaiian_ratio"), 3),
        "QUAL": safe_round(r.get("qual"), 1),
        "DP": (int(r["depth"]) if r.get("depth") is not None and not (isinstance(r["depth"],float) and math.isnan(r["depth"])) else None),
        "SIFT": r.get("sift",""),
        "PolyPhen": r.get("polyphen",""),
        "_pre_score": round(float(r.get("_pre_score", 0.0)), 3),
        "gene_knowledge": {
            "summary": short(gk.get("summary",""), 480),
            "pubmed_refs": gk.get("refs", [])
        }
    }

variants_for_llm = [format_variant_for_llm(r) for r in pref]

USER_PROMPT_MAIN = {
    "task": "Prioritize variants for dopaminergic neuron fate loss and annotate each with a narrative + confidence leveraging gene function and provided PubMed links.",
    "instructions": {
        "ranking_size": min(TOP_K, len(variants_for_llm)),
        "return_all_ranked": True,
        "notes": [
            "Use provided gene summaries and PubMed refs if present; otherwise rely on mutation class + general neuronal mechanisms.",
            "Narratives 2–5 sentences; include bracketed evidence and [GeneRef: PMID ...] when refs provided."
        ]
    },
    "schema": {
        "summary": "string",
        "most_likely": {
            "rank": 1,
            "variant_id": "string",
            "gene": "string",
            "causal_probability": 0.00,
            "confidence": 0.00,
            "rationale": "string",
            "key_evidence": ["short bullet strings"]
        },
        "ranking": [
            {
                "rank": 1,
                "variant_id": "string",
                "gene": "string",
                "causal_probability": 0.00,
                "confidence": 0.00,
                "rationale": "string",
                "key_evidence": ["short bullet strings"]
            }
        ],
        "annotations": [
            {
                "variant_id": "string",
                "gene": "string",
                "narrative": "string",
                "narrative_confidence": 0.00
            }
        ]
    },
    "variants": variants_for_llm
}

# =========================
# Call LLM (single-shot, then fallback)
# =========================
def single_shot_call() -> Optional[LLMVariantOutput]:
    temp_val = float(OPENAI_TEMPERATURE) if OPENAI_TEMPERATURE.strip() else None
    seed_val = int(OPENAI_SEED) if OPENAI_SEED.strip() else None
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_MAIN},
        {"role": "user", "content": json.dumps(USER_PROMPT_MAIN, ensure_ascii=False, separators=(",", ":"))}
    ]
    print(f"Calling {OPENAI_MODEL} with {len(variants_for_llm)} variants (single-shot)…")
    raw = openai_chat(messages, model=OPENAI_MODEL, response_json=True, temperature=temp_val, seed=seed_val, timeout=OPENAI_TIMEOUT)
    parsed = try_parse_json(raw)
    return LLMVariantOutput(**parsed)

# Fallback: chunked narratives + ranking from summaries
NARR_SYSTEM = """You are an expert in C. elegans dopaminergic neuron biology.
For each variant, write a 2–5 sentence narrative using provided fields AND gene summaries/PMIDs if present.
Cite evidence in brackets, e.g., [Effect=missense; Parental_ratio=0.82; EMS; GeneRef: PMID 12345].
Return JSON: {"annotations":[{variant_id, gene, narrative, narrative_confidence}, ...]}"""
def llm_narratives_in_chunks(items: List[Dict[str, Any]], chunk_size: int = 18) -> Dict[str, Tuple[str, float]]:
    out: Dict[str, Tuple[str, float]] = {}
    for i in range(0, len(items), chunk_size):
        chunk = items[i:i+chunk_size]
        user = {"variants": chunk, "schema": {"annotations":[{"variant_id":"string","gene":"string","narrative":"string","narrative_confidence":0.00}]}}
        messages = [
            {"role": "system", "content": NARR_SYSTEM},
            {"role": "user", "content": json.dumps(user, ensure_ascii=False, separators=(",", ":"))}
        ]
        raw = openai_chat(messages, model=OPENAI_MODEL, response_json=True, temperature=None, seed=None, timeout=OPENAI_TIMEOUT)
        parsed = try_parse_json(raw)
        anns = parsed.get("annotations", [])
        for a in anns:
            try:
                va = VariantNarrative(**a)
                out[va.variant_id] = (va.narrative, float(va.narrative_confidence))
            except ValidationError:
                pass
    return out

RANK_SYSTEM = """Rank variants for causing dopaminergic neuron fate loss using ONLY provided summaries (fields + gene refs).
Heuristics: HIGH-impact > damaging missense > low/unknown; higher Parental_ratio; EMS prior; integrate narratives & their confidence.
Output JSON only with fields: summary, most_likely, ranking[]."""
def llm_ranking_from_summaries(summaries: List[Dict[str, Any]]) -> LLMRankingOnly:
    user = {
        "task": "Rank all variants by likelihood of causing dopaminergic neuron fate loss.",
        "top_k": min(TOP_K, len(summaries)),
        "variant_summaries": summaries,
        "schema": {
            "summary": "string",
            "most_likely": {
                "rank": 1, "variant_id": "string", "gene": "string",
                "causal_probability": 0.00, "confidence": 0.00,
                "rationale": "string", "key_evidence": ["short strings"]
            },
            "ranking": [
                {"rank": 1, "variant_id": "string", "gene": "string",
                 "causal_probability": 0.00, "confidence": 0.00,
                 "rationale": "string", "key_evidence": ["short strings"]}
            ]
        }
    }
    messages = [
        {"role": "system", "content": RANK_SYSTEM},
        {"role": "user", "content": json.dumps(user, ensure_ascii=False, separators=(",", ":"))}
    ]
    raw = openai_chat(messages, model=OPENAI_MODEL, response_json=True, temperature=None, seed=None, timeout=OPENAI_TIMEOUT)
    parsed = try_parse_json(raw)
    return LLMRankingOnly(**parsed)

def chunked_fallback_pipeline() -> Tuple[pd.DataFrame, Dict[str, Tuple[str,float]]]:
    print("Falling back to chunked pipeline: generating narratives in chunks…")
    narr_map = llm_narratives_in_chunks(variants_for_llm, chunk_size=18)
    print("Chunked pipeline: ranking from compact summaries…")
    summaries = []
    for v in variants_for_llm:
        vid = v["variant_id"]
        narrative, narr_conf = narr_map.get(vid, ("", 0.0))
        summaries.append({
            "variant_id": vid,
            "gene": v.get("Gene_name",""),
            "Effect": v.get("Effect",""),
            "impact_bucket": v.get("impact_bucket","UNKNOWN"),
            "ems_label": v.get("ems_label","non-EMS"),
            "Parental_ratio": v.get("Parental_ratio", None),
            "Hawaiian_ratio": v.get("Hawaiian_ratio", None),
            "QUAL": v.get("QUAL", None),
            "DP": v.get("DP", None),
            "gene_knowledge": v.get("gene_knowledge", {}),
            "narrative": narrative,
            "narrative_confidence": narr_conf,
            "_pre_score": v.get("_pre_score", 0.0)
        })
    ranking_only = llm_ranking_from_summaries(summaries)
    rank_rows = []
    for it in sorted(ranking_only.ranking, key=lambda x: x.rank)[:TOP_K]:
        rank_rows.append({
            "run_id": RUN_ID,
            "variant_id": it.variant_id,
            "gene": it.gene,
            "rank": int(it.rank),
            "causal_probability": round(float(it.causal_probability), 2),
            "confidence": round(float(it.confidence), 2),
            "rationale": it.rationale,
            "key_evidence": "; ".join(it.key_evidence) if it.key_evidence else "",
            "Narrative": narr_map.get(it.variant_id, ("", None))[0],
            "Narrative_confidence": round(narr_map.get(it.variant_id, ("", 0.0))[1], 2) if it.variant_id in narr_map else None
        })
    return pd.DataFrame(rank_rows).sort_values(["rank","variant_id"]), narr_map

# =========================
# Execute: single-shot, else chunked fallback
# =========================
rank_df: Optional[pd.DataFrame] = None
narr_map: Dict[str, Tuple[str, float]] = {}

try:
    llm_out = single_shot_call()
    narr_map = {a.variant_id: (a.narrative, float(a.narrative_confidence)) for a in llm_out.annotations}
    ranked = sorted(llm_out.ranking, key=lambda x: x.rank)
    if TOP_K and len(ranked) > TOP_K:
        ranked = ranked[:TOP_K]
    rows = []
    for it in ranked:
        rows.append({
            "run_id": RUN_ID,
            "variant_id": it.variant_id,
            "gene": it.gene,
            "rank": int(it.rank),
            "causal_probability": round(float(it.causal_probability), 2),
            "confidence": round(float(it.confidence), 2),
            "rationale": it.rationale,
            "key_evidence": "; ".join(it.key_evidence) if it.key_evidence else "",
            "Narrative": narr_map.get(it.variant_id, ("", None))[0],
            "Narrative_confidence": round(narr_map.get(it.variant_id, ("", 0.0))[1], 2) if it.variant_id in narr_map else None
        })
    rank_df = pd.DataFrame(rows).sort_values(["rank","variant_id"])
except Exception as e:
    print(f"[Single-shot failed: {e.__class__.__name__}] {e}\n")
    rank_df, narr_map = chunked_fallback_pipeline()

# =========================
# Save artifacts
# =========================
csv_path = OUTPUT_DIR / f"llm_variant_ranking_{RUN_ID}.csv"
jsonl_path = OUTPUT_DIR / f"llm_variant_ranking_{RUN_ID}.jsonl"
rank_df.to_csv(csv_path, index=False)
with open(jsonl_path, "w", encoding="utf-8") as f:
    for _, r in rank_df.iterrows():
        f.write(json.dumps(dict(r), ensure_ascii=False) + "\n")

# Merge back selected original fields for convenient viewing (from packed_dedup)
orig_small = pd.DataFrame(packed_dedup).copy()
orig_small_ren = orig_small[[
    "variant_id","gene","effect","impact_bucket","protein_change","ems","ems_label_raw",
    "parental_ratio","hawaiian_ratio","qual","depth"
]].rename(columns={"gene": "Gene_name_csv"})

final_df = (rank_df.merge(
    orig_small_ren,
    how="left", on="variant_id"
).sort_values(["rank","variant_id"]))

# Ensure 'hawaiian_ratio' appears right after 'parental_ratio'
view_cols = [
    "rank","causal_probability","confidence",
    "variant_id","gene","Gene_name_csv","effect","impact_bucket","protein_change","ems","ems_label_raw",
    "parental_ratio","hawaiian_ratio","qual","depth",
    "Narrative","Narrative_confidence","rationale"
]
final_df = final_df[[c for c in view_cols if c in final_df.columns]]

# =========================
# Display summary
# =========================
print("\n=== Top ranked variants (preview) ===")
print(tabulate(final_df.head(20).fillna(""), headers="keys", tablefmt="github", showindex=False))

if not rank_df.empty:
    top_row = rank_df.sort_values("rank").iloc[0]
    print("\n=== Most likely causal variant (LLM) ===")
    print(tabulate([[
        int(top_row["rank"]), top_row["variant_id"], top_row["gene"],
        f"{float(top_row['causal_probability']):.2f}", f"{float(top_row['confidence']):.2f}",
        textwrap.shorten(str(narr_map.get(top_row["variant_id"], ("", 0.0))[0]), width=120),
        f"{float(narr_map.get(top_row['variant_id'], ('', 0.0))[1]):.2f}"
    ]], headers=["rank","variant_id","gene","prob","conf","narrative","narr_conf"], tablefmt="github"))

merged_path = OUTPUT_DIR / f"llm_variant_ranking_merged_{RUN_ID}.csv"
final_df.to_csv(merged_path, index=False)

print(f"\nWROTE:\n- {csv_path.resolve()}\n- {jsonl_path.resolve()}\n- {merged_path.resolve()}")
print(f"(Mapped fields from: {pathlib.Path(MAPPING_OUTPUT_PATH).resolve()})")

# =========================
# Mapping diagnostics (explicit)
# =========================
missing_counts = {
    "protein_change": int(final_df["protein_change"].isna().sum() if "protein_change" in final_df.columns else 0),
    "ems_label_raw": int(final_df["ems_label_raw"].eq("").sum() if "ems_label_raw" in final_df.columns else 0),
    "parental_ratio": int(final_df["parental_ratio"].isna().sum() if "parental_ratio" in final_df.columns else 0),
    "hawaiian_ratio": int(final_df["hawaiian_ratio"].isna().sum() if "hawaiian_ratio" in final_df.columns else 0),
    "qual": int(final_df["qual"].isna().sum() if "qual" in final_df.columns else 0),
    "depth": int(final_df["depth"].isna().sum() if "depth" in final_df.columns else 0)
}
print("\n[Post-merge diagnostics] Missing-by-column (should be near zero if mapping_output join succeeded):")
print(missing_counts)



Loaded 45 variants from: /content/gdrive/MyDrive/Code/CloudMap3/mutants/hu80/hu80_WS245_annotated-variants-mapping-region.csv
[Diagnostics] variant_id intersection: 0 ; pos_key intersection: 13
[Diagnostics] Using CHROM:POS (pos_key) to merge mapping_output fields (REF/ALT frequently missing in the annotated CSV).
Prepared base records for 16 variants after dedup by most severe effect.
Will send 16 variants to GPT‑5‑mini (FILTER_MIN_PARENTAL_RATIO=0.4, MAX_VARIANTS_TO_SEND=250).
Calling gpt-5-mini with 16 variants (single-shot)…

=== Top ranked variants (preview) ===
|   rank |   causal_probability |   confidence | variant_id      | gene       | Gene_name_csv   | effect                 | impact_bucket   | protein_change   | ems   | ems_label_raw   | parental_ratio   | hawaiian_ratio   | qual   | depth   | Narrative                                                                                                                                                                              